In [8]:
# Save all our generated data to one big file to make it easier

import torch
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
from sklearn.preprocessing import StandardScaler
import h5py

# Constants
N_LAYERS = 48
LAYERS = [f"decoder.model.decoder.layers.{n}" for n in range(N_LAYERS)]
METADATA_PATH = "/home/harinit9/orcd/pool/musicgen-data-nokey/dataset_metadata.json"
ACT_BATCHED_PATH = Path("/home/wyf/orcd/pool/musicgen-activations-nokey/activations_pooled")

In [9]:
with open(METADATA_PATH) as fin:
    metadata = json.load(fin)

N_CLIPS = len(metadata)

def get_clip_np(clip: dict):
    """
    Retrieve the activations for a single clip idx
    """
    act_path = clip["activations_path"]
    acts = torch.load(act_path)

    layer_acts = []
    for layer_idx, layer in enumerate(LAYERS):
        # acts[layer] is a list
        # shape: (256, 2048)
        layer_acts.append(torch.cat(acts[layer], dim=1).mean(axis=0).numpy())

    # shape: (N_LAYERS, 256, 2048)
    return np.stack(layer_acts)

X0 = get_clip_np(metadata[0])
print(f"{X0.shape=}")
print(f"{X0.nbytes=:,}")

X0.shape=(48, 256, 2048)
X0.nbytes=100,663,296


In [ ]:
import h5py
import numpy as np
from tqdm import tqdm

batch_size = 32

with h5py.File(ACT_BATCHED_PATH / "activations_by_layer.h5", "w") as f:
    layers_ds = [
        f.create_dataset(
            f"layer_{i}",
            shape=(N_CLIPS, 256, 2048),
            dtype=np.float32,
            compression="lzf",
            chunks=(1, 256, 2048)
        )
        for i in range(N_LAYERS)
    ]

    batch = []
    for idx, clip in enumerate(tqdm(metadata)):
        # shape: (N_LAYERS, 256, 2048)
        batch.append(get_clip_np(clip).astype(np.float32))

        if len(batch) == batch_size or idx == N_CLIPS - 1:
            start = idx + 1 - len(batch)
            # shape: (batch_size, N_LAYERS, 256, 2048)
            stacked = np.stack(batch)
            for layer_idx, ds in enumerate(layers_ds):
                ds[start:start+len(batch)] = stacked[:, layer_idx]
            
            batch = []

100%|██████████| 1000/1000 [27:14<00:00,  1.63s/it] 
